[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/tennis_racket/tennis_racket.ipynb)

# The Tennis Racket Theorem

This example simulates torque-free rigid boxes spinning near each of their principal planes across Euclidean dimensions. Spun about its plane of largest or smallest moment of inertia, a box keeps turning steadily, whereas near an intermediate plane a small disturbance grows until the box flips completely over and back again, a periodic instability known as the tennis racket theorem or Dzhanibekov effect.

Angular rate is a bivector in the turning plane and angular momentum is an antibivector linked by the duality map `points.regressive(points.commutator(Rate))`, which sums each point joined with its induced velocity ([Dorst and De Keninck, 2022](https://bivector.net/PGAdyn.pdf)). Without external forques, `motor >> inertia(rate)` stays constant in the world frame while the body-frame rate evolves by `inverse_inertia(-momentum.commutator(rate))`, and the same dimension-free expressions spin a box in every principal plane of higher-dimensional space.

In matrix Newton–Euler notation, angular rate is dualized into an axial vector $\boldsymbol{\omega} \in \mathbb{R}^3$ with inertia matrix $\mathbf{I}$ and cross-product equation $\mathbf{I}\dot{\boldsymbol{\omega}} = (\mathbf{I}\boldsymbol{\omega}) \times \boldsymbol{\omega}$ restricted to three dimensions, whereas general dimension $p$ requires $p \times p$ skew-symmetric angular-velocity matrices $\boldsymbol{\Omega} \in \mathfrak{so}(p)$ coupled through $\mathbf{J}\dot{\boldsymbol{\Omega}} + \dot{\boldsymbol{\Omega}}\mathbf{J} = [\mathbf{J}\boldsymbol{\Omega} + \boldsymbol{\Omega}\mathbf{J},\, \boldsymbol{\Omega}]$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra
from examples import instantiate
from examples.util.animation import save_figure
from examples.mechanics.tennis_racket import render, scenarios

np.set_printoptions(precision=4, suppress=True)

## Principal planes of an asymmetric box

A box of unit-mass corner points at distinct half-lengths along the coordinate axes has a distinct principal moment of inertia in every coordinate plane. Pairing the angular rate with the momentum halves into the kinetic energy `(rates & momentum) / 2`. When one copy of the box starts spinning in each principal plane with a tiny perturbation, its momentum aligns nearly with its rate so `-momentum.commutator(rates)` nearly vanishes at the start, yet only the planes of extremal moment remain stable.

In [ ]:
dimension = 3
step = 0.25
n_steps = 800
seed = 42

In [ ]:
core = instantiate("examples.mechanics.tennis_racket.core", Algebra.from_pqr(dimension, 0, 0))
motors, rates, inertia, inverse_inertia = core.racket(seed)
n_planes = len(rates)
momentum = inertia(rates)                                                       # [n_planes] Forque-
energy = (rates & momentum) / 2                                                 # [n_planes] Scalar+
# Euler's equation in the body frame maps the commutator of momentum and rate back through inverse inertia.
rate_change = inverse_inertia(-momentum.commutator(rates))                      # [n_planes] Rate+

In [ ]:
print("kinetic energy of each body:", energy.to_array())

## Tumbling in the intermediate plane

Under torque-free motion each body steps its motor by the exponential of its rate and its rate by Euler's equation. Because perturbations near the intermediate plane grow exponentially along a separatrix of the constant-energy, constant-momentum trajectories, that body periodically reverses its spin while the major-plane and minor-plane bodies hold steady.

In [ ]:
step_motors, step_rates = core.simulated_trajectory(motors, rates, inertia, inverse_inertia, core.lie.explicit_rkmk4, step, n_steps)
energies = core.lie.kinetic_energy(step_rates, inertia)                         # [n_steps + 1, n_planes] Scalar+
# Carry body-frame angular momentum out to the world frame with each step's motor.
world_momenta = step_motors >> inertia(step_rates)                              # [n_steps + 1, n_planes] Forque-
# Pairing the rate with the initial angular momentum goes negative when a body's spin plane flips over.
spin_flips = ((step_rates & momentum) < 0).any(axis=0)
save_figure(render.trajectories_figure(step_rates, energies), "tennis_racket_3d");

Each panel shows one body labelled by its conserved kinetic energy, with the three curves tracing the components of its rate in the three coordinate planes; the top and bottom panels hold flat while the middle panel flips cleanly between positive and negative spin.

In [ ]:
print("spin flipped, per body:", spin_flips)

## Six principal planes in four dimensions

A four-dimensional box with four distinct half-lengths has six coordinate planes, and instantiating the same core in four-dimensional Euclidean space spins one box in each plane.

In [ ]:
higher_dimension = 4

In [ ]:
higher_rates, higher_energies = scenarios.spinning_racket(higher_dimension, step, n_steps * step, seed)
save_figure(render.trajectories_figure(higher_rates, higher_energies), "tennis_racket");

Across the six panels, the planes of smallest and largest kinetic energy again spin steadily, while the four intermediate planes split between steady rotation, regular flips, and irregular multi-plane tumbling.

## World-frame momentum conservation across integrators

Stepping the body rate by Runge–Kutta and updating the motor by a single exponential of the stepped rate, as explicit Verlet and naive RK4 do, lets world-frame angular momentum drift at first order because non-commuting turns curve the rotor manifold. The Runge–Kutta–Munthe-Kaas method integrates a bivector increment in the Lie algebra instead, correcting the rate for the curvature of the exponential map via `h.commutator(Bivector)` so that world momentum stays conserved to fourth order in every dimension.

In the notation of Lie-group numerical analysis, that curvature correction evaluates the truncated inverse differential of the exponential map, $\operatorname{dexp}^{-1}_h = I - \tfrac{1}{2}\operatorname{ad}_h + \tfrac{1}{12}\operatorname{ad}_h^2$.

In [ ]:
compared_dimensions = (3, 4, 5)
comparison_runtime = 100.0

In [ ]:
curves = list(scenarios.integrator_comparison(compared_dimensions, step, comparison_runtime, seed))
save_figure(render.integrator_comparison_figure(curves, step), "tennis_racket_integrators");

In each dimension panel (`3D`, `4D`, and `5D`), the upper solid and dashed curves show first-order drift under explicit Verlet and naive RK4, while the bottom solid curve shows Munthe-Kaas holding world-frame angular momentum roughly four orders of magnitude tighter.

In [ ]:
for compared_dimension, name, drift in curves:
    print(f"{compared_dimension} dimensions, {name:6s}: worst world momentum drift {drift.max().to_array():.2e}")

In [ ]:
# checks
# Principal-plane spin starts with near-zero rate change, only the intermediate plane flips in 3D,
# kinetic energy stays conserved, and Munthe-Kaas preserves world momentum across all dimensions.
assert (rate_change.norm() < 1e-3).all()
assert spin_flips.sum() == 1
assert ((world_momenta - world_momenta[0]).antinorm() / world_momenta[0].antinorm() < 1e-4).all()
assert ((energies / energies[0] - 1).abs() < 1e-4).all()
worst = {(compared_dimension, name): drift.max() for compared_dimension, name, drift in curves}
for compared_dimension in compared_dimensions:
    assert worst[compared_dimension, "rkmk4"] < 0.1 * worst[compared_dimension, "verlet"]